# 02_05 — Limpieza de `contexto_calendario_laboral`

## Objetivo

Este notebook limpia la fuente `contexto_calendario_laboral` para construir una tabla diaria fiable que permita segmentar días comparables y validar posteriormente el régimen observable del SER.

Esta fuente no mide aparcamiento ni ocupación. Su función es temporal: distinguir laborables, sábados, domingos y festivos para evitar comparaciones incorrectas entre días con regímenes de uso distintos.

## Uso posterior en el TFM

La tabla limpia se usará para:

- validar días con posible régimen SER observable;
- explicar caídas o ausencia de tiques en domingos y festivos;
- segmentar análisis de SER, EMT, tráfico y clima por tipo de día;
- cruzar por `fecha` con tablas horarias o diarias posteriores.

No se construyen todavía joins con SER ni métricas de dificultad.

## Referencia documental

La estructura esperada de `contexto_calendario_laboral` se toma de la documentación oficial del conjunto de datos, guardada en:

`docs/source_docs/contexto/contexto_calendario_laboral/300082-4-calendario_laboral.pdf`

Según esa documentación, el dataset contiene cinco campos: `Fecha`, `Dia_semana`, `Laborable/festivo/domingo festivo`, `Tipo de festivo` y `Festividad`.

En el fichero CSV real, la columna oficial `Fecha` aparece como `Dia`. Por tanto, en esta limpieza se interpreta `dia` como la fecha calendario del registro, no como día del mes. Esta decisión se valida posteriormente comprobando el parseo de fechas, la cobertura temporal, la coherencia con el día de semana y la ausencia de duplicados por fecha.

Esta fuente se usa como dimensión temporal diaria auxiliar. No mide aparcamiento ni ocupación, pero permite segmentar días comparables y validar posteriormente cuándo la señal de tiques SER debería ser observable.


In [105]:
from pathlib import Path
import re
import unicodedata
import json
import numpy as np
import pandas as pd
from IPython.display import display

DATASET_ID = "contexto_calendario_laboral"

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "data_catalog.csv").exists():
    ROOT = ROOT.parent

if not (ROOT / "data_catalog.csv").exists():
    raise FileNotFoundError(
        "No se ha encontrado data_catalog.csv. Ejecuta el notebook desde dentro del repo TFM_parking_madrid."
    )

RAW_DIR = ROOT / "data" / "raw" / "contexto" / DATASET_ID
OUT_DIR = ROOT / "data" / "interim" / "contexto" / DATASET_ID
REPORT_REVIEW_DIR = ROOT / "reports" / "review"
REPORT_TABLES_DIR = ROOT / "reports" / "tables"
DOCS_LIMPIEZA_DIR = ROOT / "docs" / "limpieza"

OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_REVIEW_DIR.mkdir(parents=True, exist_ok=True)
REPORT_TABLES_DIR.mkdir(parents=True, exist_ok=True)
DOCS_LIMPIEZA_DIR.mkdir(parents=True, exist_ok=True)

OUT_PARQUET = OUT_DIR / "contexto_calendario_laboral_clean.parquet"
QUALITY_CSV = REPORT_TABLES_DIR / "contexto_calendario_laboral_quality_checks.csv"
INTERPRETACION_MD = DOCS_LIMPIEZA_DIR / "contexto_calendario_laboral.md"

ANIO_MIN_TFM = 2023
ANIO_MAX_TFM = 2026

print("ROOT:", ROOT)
print("RAW_DIR:", RAW_DIR)
print("OUT_PARQUET:", OUT_PARQUET)

ROOT: /Users/hugo/TFM_parking_madrid
RAW_DIR: /Users/hugo/TFM_parking_madrid/data/raw/contexto/contexto_calendario_laboral
OUT_PARQUET: /Users/hugo/TFM_parking_madrid/data/interim/contexto/contexto_calendario_laboral/contexto_calendario_laboral_clean.parquet


## 1. Funciones auxiliares

Se normalizan nombres de columnas y valores de texto para evitar errores por acentos, espacios, mayúsculas o separadores distintos.

Esto es necesario porque los ficheros municipales pueden cambiar ligeramente nombres de columnas o formatos entre versiones.

In [106]:
def normalize_colname(name: str) -> str:
    """Normaliza nombres de columnas a snake_case sin acentos."""
    name = str(name).strip()
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    name = name.lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


def normalize_text_value(value):
    """Normaliza texto para comparaciones robustas; mantiene NA si el valor está vacío."""
    if pd.isna(value):
        return pd.NA
    value = str(value).strip()
    if value == "":
        return pd.NA
    value = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode("ascii")
    value = value.lower().strip()
    value = re.sub(r"\s+", " ", value)
    return value


def clean_string_series(s: pd.Series) -> pd.Series:
    """Limpia strings preservando valores nulos."""
    return (
        s.astype("string")
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    )


def normalize_separator(value):
    """Convierte separadores descritos en data_catalog a separadores reales de pandas."""
    if pd.isna(value):
        return None

    value = str(value).strip()

    mapping = {
        "semicolon": ";",
        "punto_y_coma": ";",
        "punto y coma": ";",
        "comma": ",",
        "coma": ",",
        "tab": "\t",
        "tabulador": "\t",
        "pipe": "|",
    }

    return mapping.get(value.lower(), value)


def read_csv_robust(path: Path, sep=None, encoding=None) -> pd.DataFrame:
    """Lectura robusta de CSV probando separador y codificación si no vienen cerrados."""
    sep = normalize_separator(sep)

    encodings = [encoding] if encoding else ["utf-8-sig", "utf-8", "latin1", "cp1252"]
    seps = [sep] if sep else [";", ",", "\t", "|"]

    errors = []

    for enc in encodings:
        for s in seps:
            try:
                return pd.read_csv(path, sep=s, encoding=enc, low_memory=False)
            except Exception as e:
                errors.append((enc, s, str(e)[:200]))

    for enc in encodings:
        try:
            return pd.read_csv(path, sep=None, engine="python", encoding=enc)
        except Exception as e:
            errors.append((enc, "infer", str(e)[:200]))

    raise RuntimeError(f"No se pudo leer {path}. Errores: {errors[:8]}")


def find_column(columns, required_tokens=None, any_tokens=None):
    """Busca una columna por tokens obligatorios y opcionales."""
    required_tokens = required_tokens or []
    any_tokens = any_tokens or []

    matches = []
    for col in columns:
        ok_required = all(tok in col for tok in required_tokens)
        ok_any = True if not any_tokens else any(tok in col for tok in any_tokens)

        if ok_required and ok_any:
            matches.append(col)

    if not matches:
        return None

    return sorted(matches, key=len)[0]


def df_to_markdown_safe(df: pd.DataFrame) -> str:
    """Convierte a markdown si tabulate está instalado; si no, usa texto plano."""
    try:
        return df.to_markdown(index=False)
    except Exception:
        return "```text\n" + df.to_string(index=False) + "\n```"

## 2. Carga desde `data_catalog.csv`

La fuente se localiza desde `data_catalog.csv` para mantener trazabilidad entre `dataset_id`, ruta local y uso dentro del TFM.

Si el catálogo no apunta directamente a un CSV existente, se hace fallback buscando ficheros `.csv` dentro de `data/raw/contexto/contexto_calendario_laboral/`.

In [107]:
catalog = pd.read_csv(ROOT / "data_catalog.csv")
catalog.columns = [normalize_colname(c) for c in catalog.columns]

if "dataset_id" not in catalog.columns:
    raise ValueError("data_catalog.csv debe contener una columna dataset_id.")

source_rows = catalog[catalog["dataset_id"].astype(str).str.strip().eq(DATASET_ID)].copy()

if source_rows.empty:
    raise ValueError(f"No se ha encontrado dataset_id={DATASET_ID} en data_catalog.csv.")

print("Filas encontradas en data_catalog:", len(source_rows))
display(source_rows)

Filas encontradas en data_catalog: 1


,dataset_id,bloque,prioridad,nombre_fuente,source_code,url_fuente,tipo_acceso,formato,formato_preferido,periodo_dato_objetivo,archivo_raw,archivo_interim,unidad_espacial,granularidad_temporal,estado
9,contexto_calendario_laboral,contexto,complementaria_v1,Calendario laboral,300082,https://datos.madrid.es/dataset/300082-0-calen...,manual,CSV/ICS,csv,2023-2026,data/raw/contexto/contexto_calendario_laboral/...,data/interim/contexto/contexto_calendario_labo...,ciudad_dia,dia,raw_descargado


In [108]:
def candidate_paths_from_catalog(row: pd.Series) -> list[Path]:
    """Extrae posibles rutas locales desde columnas tipo path/file del catálogo."""
    candidates = []

    path_like_cols = [
        c for c in row.index
        if any(token in c for token in ["path", "ruta", "archivo", "file", "filename", "physical"])
    ]

    for col in path_like_cols:
        value = row.get(col)
        if pd.isna(value):
            continue

        value = str(value).strip()
        if not value or value.lower() in {"nan", "none"}:
            continue

        p = Path(value)

        if p.is_absolute():
            candidates.append(p)
        else:
            candidates.append(ROOT / p)
            candidates.append(RAW_DIR / p.name)

    return candidates


candidate_paths = []
for _, row in source_rows.iterrows():
    candidate_paths.extend(candidate_paths_from_catalog(row))

candidate_paths = [p for p in candidate_paths if p.exists() and p.suffix.lower() == ".csv"]

if not candidate_paths:
    candidate_paths = sorted(RAW_DIR.rglob("*.csv"))

if not candidate_paths:
    raise FileNotFoundError(
        f"No se ha encontrado ningún CSV para {DATASET_ID}. Revisa {RAW_DIR} y data_catalog.csv."
    )

candidate_paths = sorted(set(candidate_paths))

print("CSV encontrados:")
for p in candidate_paths:
    print("-", p)

CSV encontrados:
- /Users/hugo/TFM_parking_madrid/data/raw/contexto/contexto_calendario_laboral/contexto_calendario_laboral__2013_2026.csv


In [109]:
raw_frames = []

for path in candidate_paths:
    row_meta = source_rows.iloc[0]

    sep = row_meta.get("separator", None)
    encoding = row_meta.get("encoding", None)

    sep = None if pd.isna(sep) else str(sep)
    encoding = None if pd.isna(encoding) else str(encoding)

    df_part = read_csv_robust(path, sep=sep, encoding=encoding)
    df_part["archivo_origen"] = str(path.relative_to(ROOT))
    raw_frames.append(df_part)

df_raw = pd.concat(raw_frames, ignore_index=True)

print(f"Filas raw: {len(df_raw):,}")
print(f"Columnas raw: {list(df_raw.columns)}")
display(df_raw.head())

Filas raw: 5,112
Columnas raw: ['Dia', 'Dia_semana', 'laborable / festivo / domingo festivo', 'Tipo de Festivo', 'Festividad', 'archivo_origen']


,Dia,Dia_semana,laborable / festivo / domingo festivo,Tipo de Festivo,Festividad,archivo_origen
0,01/01/2013,martes,festivo,Festivo nacional,Año Nuevo,data/raw/contexto/contexto_calendario_laboral/...
1,02/01/2013,miercoles,laborable,NaN,NaN,data/raw/contexto/contexto_calendario_laboral/...
2,03/01/2013,jueves,laborable,NaN,NaN,data/raw/contexto/contexto_calendario_laboral/...
3,04/01/2013,viernes,laborable,NaN,NaN,data/raw/contexto/contexto_calendario_laboral/...
4,05/01/2013,sabado,sabado,NaN,NaN,data/raw/contexto/contexto_calendario_laboral/...


## 3. Normalización de columnas

Se normalizan los nombres de columnas para trabajar con una convención estable.

El objetivo no es alterar el contenido, sino reducir fragilidad en el código posterior.

In [110]:
df = df_raw.copy()
df.columns = [normalize_colname(c) for c in df.columns]

print("Columnas normalizadas:")
print(df.columns.tolist())

display(df.head())

Columnas normalizadas:
['dia', 'dia_semana', 'laborable_festivo_domingo_festivo', 'tipo_de_festivo', 'festividad', 'archivo_origen']


,dia,dia_semana,laborable_festivo_domingo_festivo,tipo_de_festivo,festividad,archivo_origen
0,01/01/2013,martes,festivo,Festivo nacional,Año Nuevo,data/raw/contexto/contexto_calendario_laboral/...
1,02/01/2013,miercoles,laborable,NaN,NaN,data/raw/contexto/contexto_calendario_laboral/...
2,03/01/2013,jueves,laborable,NaN,NaN,data/raw/contexto/contexto_calendario_laboral/...
3,04/01/2013,viernes,laborable,NaN,NaN,data/raw/contexto/contexto_calendario_laboral/...
4,05/01/2013,sabado,sabado,NaN,NaN,data/raw/contexto/contexto_calendario_laboral/...


In [111]:
# Detección robusta de columnas clave.
# La documentación oficial llama "Fecha" a la primera columna, pero el CSV real puede traerla como "Dia".
# En este dataset, "dia" no se interpreta como día del mes: se interpreta como fecha calendario si parsea como fecha.

def detect_date_column(df: pd.DataFrame) -> str | None:
    """Detecta la columna de fecha aceptando 'fecha' o 'dia' si parsea mayoritariamente como fecha."""
    candidates = []

    for c in df.columns:
        c_norm = normalize_colname(c)
        if "fecha" in c_norm:
            candidates.append(c)

    # Caso real del calendario laboral: columna 'dia' con valores tipo dd/mm/aaaa.
    if "dia" in df.columns:
        candidates.append("dia")

    # Evitar duplicados manteniendo orden.
    seen = set()
    candidates = [c for c in candidates if not (c in seen or seen.add(c))]

    scored = []
    for c in candidates:
        parsed = pd.to_datetime(df[c], errors="coerce", dayfirst=True)
        valid_ratio = parsed.notna().mean()
        scored.append((c, valid_ratio))

    if not scored:
        return None

    scored = sorted(scored, key=lambda x: x[1], reverse=True)
    best_col, best_ratio = scored[0]

    if best_ratio < 0.90:
        print("Candidatas de fecha evaluadas:", scored)
        return None

    print("Candidatas de fecha evaluadas:", scored)
    print(f"Columna de fecha seleccionada: {best_col} (ratio parseo válido={best_ratio:.3f})")
    return best_col


col_fecha = detect_date_column(df)
col_dia_semana = find_column(df.columns, required_tokens=["dia"], any_tokens=["semana"])

# Búsqueda más estricta para evitar confundir "tipo_de_festivo" con la columna principal de tipo de día.
col_tipo_dia = None

for c in df.columns:
    if "laborable" in c:
        col_tipo_dia = c
        break

if col_tipo_dia is None:
    for c in df.columns:
        c_norm = normalize_colname(c)
        if "domingo" in c_norm and "festivo" in c_norm and "tipo" not in c_norm:
            col_tipo_dia = c
            break

if col_tipo_dia is None:
    col_tipo_dia = find_column(df.columns, any_tokens=["laborable", "festivo", "domingo"])

col_tipo_festivo = find_column(df.columns, required_tokens=["tipo"], any_tokens=["festivo"])
col_festividad = find_column(df.columns, any_tokens=["festividad"])

detected_cols = {
    "fecha": col_fecha,
    "dia_semana_raw": col_dia_semana,
    "tipo_dia_raw": col_tipo_dia,
    "tipo_festivo": col_tipo_festivo,
    "festividad": col_festividad,
}

detected_cols


Candidatas de fecha evaluadas: [('dia', 1.0)]
Columna de fecha seleccionada: dia (ratio parseo válido=1.000)


{'fecha': 'dia',
 'dia_semana_raw': 'dia_semana',
 'tipo_dia_raw': 'laborable_festivo_domingo_festivo',
 'tipo_festivo': 'tipo_de_festivo',
 'festividad': 'festividad'}

In [112]:
required_detected = ["fecha", "tipo_dia_raw"]

missing = [k for k in required_detected if detected_cols[k] is None]
if missing:
    raise ValueError(
        f"No se han detectado columnas mínimas {missing}. "
        f"Columnas disponibles: {df.columns.tolist()}"
    )

## 4. Construcción de tabla limpia diaria

Se crea una tabla con una fila por fecha y variables temporales derivadas desde la fecha real.

La columna de día de semana se recalcula desde `fecha`, no se toma ciegamente del CSV. Esto permite validar la coherencia del fichero raw.

In [113]:
clean = pd.DataFrame()

clean["fecha"] = pd.to_datetime(df[col_fecha], errors="coerce", dayfirst=True).dt.normalize()

clean["anio"] = clean["fecha"].dt.year.astype("Int64")
clean["mes"] = clean["fecha"].dt.month.astype("Int64")
clean["dia"] = clean["fecha"].dt.day.astype("Int64")
clean["dia_semana_num"] = clean["fecha"].dt.dayofweek.add(1).astype("Int64")

dia_semana_map = {
    1: "lunes",
    2: "martes",
    3: "miércoles",
    4: "jueves",
    5: "viernes",
    6: "sábado",
    7: "domingo",
}
clean["dia_semana_nombre"] = clean["dia_semana_num"].map(dia_semana_map).astype("string")

if col_dia_semana is not None:
    clean["dia_semana_raw"] = clean_string_series(df[col_dia_semana])
else:
    clean["dia_semana_raw"] = pd.Series(pd.NA, index=clean.index, dtype="string")

clean["tipo_dia_raw"] = clean_string_series(df[col_tipo_dia])

if col_tipo_festivo is not None:
    clean["tipo_festivo"] = clean_string_series(df[col_tipo_festivo])
else:
    clean["tipo_festivo"] = pd.Series(pd.NA, index=clean.index, dtype="string")

if col_festividad is not None:
    clean["festividad"] = clean_string_series(df[col_festividad])
else:
    clean["festividad"] = pd.Series(pd.NA, index=clean.index, dtype="string")

clean["archivo_origen"] = df["archivo_origen"].astype("string")
clean["dataset_id"] = DATASET_ID

display(clean.head())

,fecha,anio,mes,dia,dia_semana_num,dia_semana_nombre,dia_semana_raw,tipo_dia_raw,tipo_festivo,festividad,archivo_origen,dataset_id
0,2013-01-01,2013,1,1,2,martes,martes,festivo,Festivo nacional,Año Nuevo,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral
1,2013-01-02,2013,1,2,3,miércoles,miercoles,laborable,<NA>,<NA>,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral
2,2013-01-03,2013,1,3,4,jueves,jueves,laborable,<NA>,<NA>,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral
3,2013-01-04,2013,1,4,5,viernes,viernes,laborable,<NA>,<NA>,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral
4,2013-01-05,2013,1,5,6,sábado,sabado,sabado,<NA>,<NA>,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral


## 4.1. Filtrado de ventana analítica

El fichero raw contiene una cobertura más amplia que la necesaria para este TFM. Aunque se conserva constancia de la cobertura raw, la tabla limpia final se restringe a 2023–2026, que es la ventana temporal objetivo del proyecto.

Este filtrado evita arrastrar anomalías históricas ajenas al periodo de análisis y mantiene coherencia con el resto de fuentes SER, EMT y contexto que se trabajarán en la fase actual.

In [114]:
# Conservamos una copia diagnóstica de la cobertura raw antes de filtrar.
clean_raw_full = clean.copy()

raw_fecha_min = clean_raw_full["fecha"].min()
raw_fecha_max = clean_raw_full["fecha"].max()
raw_n_filas = len(clean_raw_full)
raw_anios = sorted(clean_raw_full["anio"].dropna().astype(int).unique().tolist())

print("Cobertura raw cargada:")
print(f"- filas raw limpias antes de filtrar: {raw_n_filas:,}")
print(f"- fecha mínima raw: {raw_fecha_min}")
print(f"- fecha máxima raw: {raw_fecha_max}")
print(f"- años raw: {raw_anios}")

# Filtro de ventana analítica del TFM.
mask_window_tfm = clean["anio"].between(ANIO_MIN_TFM, ANIO_MAX_TFM)
clean = clean.loc[mask_window_tfm].copy().reset_index(drop=True)

print("\nCobertura analítica usada en el TFM:")
print(f"- filas tras filtro {ANIO_MIN_TFM}-{ANIO_MAX_TFM}: {len(clean):,}")
print(f"- fecha mínima filtrada: {clean['fecha'].min()}")
print(f"- fecha máxima filtrada: {clean['fecha'].max()}")
print(f"- años filtrados: {sorted(clean['anio'].dropna().astype(int).unique().tolist())}")

if clean.empty:
    raise ValueError("El filtrado 2023-2026 ha dejado la tabla vacía.")

if clean["anio"].min() < ANIO_MIN_TFM or clean["anio"].max() > ANIO_MAX_TFM:
    raise ValueError("El filtro temporal no se ha aplicado correctamente.")

expected_n_days = len(pd.date_range(
    start=f"{ANIO_MIN_TFM}-01-01",
    end=f"{ANIO_MAX_TFM}-12-31",
    freq="D",
))

if len(clean) != expected_n_days:
    print(f"AVISO: se esperaban {expected_n_days} días en la ventana, pero hay {len(clean)} filas.")

display(clean.head())
display(clean.tail())

Cobertura raw cargada:
- filas raw limpias antes de filtrar: 5,112
- fecha mínima raw: 2013-01-01 00:00:00
- fecha máxima raw: 2026-12-31 00:00:00
- años raw: [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

Cobertura analítica usada en el TFM:
- filas tras filtro 2023-2026: 1,461
- fecha mínima filtrada: 2023-01-01 00:00:00
- fecha máxima filtrada: 2026-12-31 00:00:00
- años filtrados: [2023, 2024, 2025, 2026]


,fecha,anio,mes,dia,dia_semana_num,dia_semana_nombre,dia_semana_raw,tipo_dia_raw,tipo_festivo,festividad,archivo_origen,dataset_id
0,2023-01-01,2023,1,1,7,domingo,domingo,domingo,<NA>,<NA>,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral
1,2023-01-02,2023,1,2,1,lunes,lunes,laborable,<NA>,<NA>,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral
2,2023-01-03,2023,1,3,2,martes,martes,laborable,<NA>,<NA>,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral
3,2023-01-04,2023,1,4,3,miércoles,miercoles,laborable,<NA>,<NA>,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral
4,2023-01-05,2023,1,5,4,jueves,jueves,laborable,<NA>,<NA>,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral


,fecha,anio,mes,dia,dia_semana_num,dia_semana_nombre,dia_semana_raw,tipo_dia_raw,tipo_festivo,festividad,archivo_origen,dataset_id
1456,2026-12-27,2026,12,27,7,domingo,domingo,domingo,<NA>,<NA>,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral
1457,2026-12-28,2026,12,28,1,lunes,lunes,laborable,<NA>,<NA>,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral
1458,2026-12-29,2026,12,29,2,martes,martes,laborable,<NA>,<NA>,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral
1459,2026-12-30,2026,12,30,3,miércoles,miercoles,laborable,<NA>,<NA>,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral
1460,2026-12-31,2026,12,31,4,jueves,jueves,laborable,<NA>,<NA>,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral


## 5. Derivación de flags temporales

Se derivan los flags principales:

- `es_laborable`
- `es_sabado`
- `es_domingo`
- `es_festivo`

Además, se crea `es_dia_potencial_ser_observable`, que solo debe interpretarse como una ayuda diaria. El régimen SER completo depende también de la hora, por lo que se validará después en `ser_tiques`.

In [115]:
tipo_norm = clean["tipo_dia_raw"].map(normalize_text_value).astype("string")
tipo_festivo_norm = clean["tipo_festivo"].map(normalize_text_value).astype("string")
festividad_norm = clean["festividad"].map(normalize_text_value).astype("string")

clean["es_sabado"] = clean["dia_semana_num"].eq(6)
clean["es_domingo"] = clean["dia_semana_num"].eq(7)

clean["es_festivo"] = (
    tipo_norm.str.contains("festivo", na=False)
    | tipo_festivo_norm.notna()
    | festividad_norm.notna()
)

clean["es_laborable"] = (
    tipo_norm.eq("laborable")
    & ~clean["es_sabado"]
    & ~clean["es_domingo"]
    & ~clean["es_festivo"]
)

clean["es_dia_potencial_ser_observable"] = (
    (clean["es_laborable"] | clean["es_sabado"])
    & ~clean["es_domingo"]
    & ~clean["es_festivo"]
)

clean["tipo_regimen_ser_dia_base"] = np.select(
    [
        clean["es_festivo"] & clean["es_domingo"],
        clean["es_festivo"],
        clean["es_domingo"],
        clean["es_sabado"],
        clean["es_laborable"],
    ],
    [
        "domingo_festivo",
        "festivo",
        "domingo",
        "sabado",
        "laborable",
    ],
    default="otro_no_clasificado",
)

display(clean.head())

,fecha,anio,mes,dia,dia_semana_num,dia_semana_nombre,dia_semana_raw,tipo_dia_raw,tipo_festivo,festividad,archivo_origen,dataset_id,es_sabado,es_domingo,es_festivo,es_laborable,es_dia_potencial_ser_observable,tipo_regimen_ser_dia_base
0,2023-01-01,2023,1,1,7,domingo,domingo,domingo,<NA>,<NA>,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral,False,True,False,False,False,domingo
1,2023-01-02,2023,1,2,1,lunes,lunes,laborable,<NA>,<NA>,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral,False,False,False,True,True,laborable
2,2023-01-03,2023,1,3,2,martes,martes,laborable,<NA>,<NA>,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral,False,False,False,True,True,laborable
3,2023-01-04,2023,1,4,3,miércoles,miercoles,laborable,<NA>,<NA>,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral,False,False,False,True,True,laborable
4,2023-01-05,2023,1,5,4,jueves,jueves,laborable,<NA>,<NA>,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral,False,False,False,True,True,laborable


## 6. Validaciones de calidad

Estas validaciones comprueban que la tabla puede funcionar como dimensión temporal diaria:

- fechas nulas;
- duplicados por fecha;
- cobertura 2023–2026;
- fechas faltantes dentro del periodo objetivo;
- coherencia entre fecha real y día de semana declarado;
- valores únicos de tipo de día;
- domingos correctamente marcados;
- festivos identificados;
- incompatibilidades lógicas.

In [116]:

checks = []

def add_check(nombre, valor, ok, detalle=""):
    checks.append({
        "check": nombre,
        "valor": valor,
        "ok": bool(ok),
        "detalle": detalle,
    })


# 1. Fechas nulas
n_fecha_nula = int(clean["fecha"].isna().sum())
add_check(
    "fechas_nulas",
    n_fecha_nula,
    n_fecha_nula == 0,
    "Debe ser 0 para poder usar la tabla como dimensión temporal diaria."
)

# 2. Duplicados por fecha
n_dup_fecha = int(clean.duplicated(subset=["fecha"], keep=False).sum())
add_check(
    "duplicados_por_fecha",
    n_dup_fecha,
    n_dup_fecha == 0,
    "Debe haber una única fila por fecha."
)

# 3. Cobertura 2023-2026
years_present = sorted(clean["anio"].dropna().astype(int).unique().tolist())
expected_years = list(range(ANIO_MIN_TFM, ANIO_MAX_TFM + 1))
missing_years = sorted(set(expected_years) - set(years_present))

add_check(
    "cobertura_anios_2023_2026",
    json.dumps({"presentes": years_present, "faltantes": missing_years}, ensure_ascii=False),
    len(missing_years) == 0,
    "La salida limpia debe cubrir exactamente la ventana 2023-2026."
)

# 4. Fechas faltantes dentro de 2023-2026
calendar_expected = pd.date_range(
    start=f"{ANIO_MIN_TFM}-01-01",
    end=f"{ANIO_MAX_TFM}-12-31",
    freq="D",
)

fechas_presentes_tfm = set(clean["fecha"].dropna())
missing_dates = sorted(set(calendar_expected) - fechas_presentes_tfm)

add_check(
    "fechas_faltantes_2023_2026",
    len(missing_dates),
    len(missing_dates) == 0,
    f"Primeras fechas faltantes: {[str(d.date()) for d in missing_dates[:10]]}"
)

# 5. Coherencia fecha real vs día de semana declarado en el raw.
# Importante: se usa clean["dia_semana_raw"], ya filtrado junto con clean.
if "dia_semana_raw" in clean.columns:
    raw_weekday_norm = clean["dia_semana_raw"].map(normalize_text_value).astype("string")

    weekday_num_from_raw = raw_weekday_norm.map({
        "lunes": 1,
        "martes": 2,
        "miercoles": 3,
        "miércoles": 3,
        "jueves": 4,
        "viernes": 5,
        "sabado": 6,
        "sábado": 6,
        "domingo": 7,
    }).astype("Int64")

    weekday_mismatch = (
        weekday_num_from_raw.notna()
        & clean["dia_semana_num"].notna()
        & weekday_num_from_raw.ne(clean["dia_semana_num"])
    )

    n_weekday_mismatch = int(weekday_mismatch.sum())
    add_check(
        "coherencia_fecha_vs_dia_semana_raw",
        n_weekday_mismatch,
        n_weekday_mismatch == 0,
        "Compara el día de semana declarado en raw, ya filtrado, con el calculado desde fecha."
    )
else:
    weekday_mismatch = pd.Series(False, index=clean.index)
    add_check(
        "coherencia_fecha_vs_dia_semana_raw",
        "no_aplica",
        True,
        "No se conserva columna raw de día de semana."
    )

# 6. Valores únicos de tipo de día
tipo_dia_unicos = sorted(clean["tipo_dia_raw"].dropna().astype(str).unique().tolist())
add_check(
    "valores_unicos_tipo_dia_raw",
    json.dumps(tipo_dia_unicos, ensure_ascii=False),
    len(tipo_dia_unicos) > 0,
    "Debe revisarse que aparecen laborable, sábado, domingo y festivo si proceden."
)

# 7. Domingos reales marcados como domingo
tipo_norm = clean["tipo_dia_raw"].map(normalize_text_value).astype("string")
domingos_mal_marcados = clean["es_domingo"] & ~tipo_norm.str.contains("domingo", na=False)

n_domingos_mal = int(domingos_mal_marcados.sum())
add_check(
    "domingos_marcados_como_domingo",
    n_domingos_mal,
    n_domingos_mal == 0,
    "Todo domingo real debería aparecer marcado como domingo o domingo festivo en tipo_dia_raw."
)

# 8. Festivos
n_festivos = int(clean["es_festivo"].sum())
n_festivos_sin_detalle = int(
    (
        clean["es_festivo"]
        & clean["tipo_festivo"].isna()
        & clean["festividad"].isna()
    ).sum()
)

add_check(
    "festivos_detectados",
    n_festivos,
    n_festivos > 0,
    "Debe haber festivos identificados si el calendario cubre años completos."
)

add_check(
    "festivos_sin_tipo_ni_festividad",
    n_festivos_sin_detalle,
    n_festivos_sin_detalle == 0,
    "Si es >0, hay festivos detectados solo por tipo_dia_raw sin detalle adicional."
)

# 9. Incompatibilidades lógicas
n_laborable_y_festivo = int((clean["es_laborable"] & clean["es_festivo"]).sum())
add_check(
    "laborable_y_festivo_a_la_vez",
    n_laborable_y_festivo,
    n_laborable_y_festivo == 0,
    "Una fecha no debería ser laborable y festiva simultáneamente."
)

n_sabado_y_domingo = int((clean["es_sabado"] & clean["es_domingo"]).sum())
add_check(
    "sabado_y_domingo_a_la_vez",
    n_sabado_y_domingo,
    n_sabado_y_domingo == 0,
    "Una fecha no puede ser sábado y domingo simultáneamente."
)

quality = pd.DataFrame(checks)

checks_criticos = [
    "fechas_nulas",
    "duplicados_por_fecha",
    "cobertura_anios_2023_2026",
    "fechas_faltantes_2023_2026",
    "coherencia_fecha_vs_dia_semana_raw",
    "valores_unicos_tipo_dia_raw",
    "festivos_detectados",
    "laborable_y_festivo_a_la_vez",
    "sabado_y_domingo_a_la_vez",
]

checks_aviso = [
    "domingos_marcados_como_domingo",
    "festivos_sin_tipo_ni_festividad",
]

quality["nivel"] = np.where(
    quality["check"].isin(checks_criticos),
    "critico",
    "aviso"
)

checks_criticos_ok = bool(quality.loc[quality["nivel"].eq("critico"), "ok"].all())
checks_aviso_ok = bool(quality.loc[quality["nivel"].eq("aviso"), "ok"].all())

# Diagnóstico adicional solo si hay anomalías dentro de la ventana analítica
mask_domingo_real_no_domingo_raw = domingos_mal_marcados
mask_festivo_sin_detalle = (
    clean["es_festivo"]
    & clean["tipo_festivo"].isna()
    & clean["festividad"].isna()
)

cols_diag = [
    "fecha",
    "anio",
    "mes",
    "dia",
    "dia_semana_num",
    "dia_semana_nombre",
    "tipo_dia_raw",
    "tipo_festivo",
    "festividad",
    "es_domingo",
    "es_festivo",
    "tipo_regimen_ser_dia_base",
    "archivo_origen",
]

diagnostico_domingo = clean.loc[mask_domingo_real_no_domingo_raw, cols_diag].copy()
diagnostico_domingo["anomalia"] = "domingo_real_no_marcado_explictamente_como_domingo"

diagnostico_festivo = clean.loc[mask_festivo_sin_detalle, cols_diag].copy()
diagnostico_festivo["anomalia"] = "festivo_sin_tipo_ni_festividad"

diagnostico_anomalias = pd.concat(
    [diagnostico_domingo, diagnostico_festivo],
    ignore_index=True
).sort_values(["fecha", "anomalia"])

DIAG_CSV = REPORT_TABLES_DIR / "contexto_calendario_laboral_diagnostico_anomalias.csv"

if diagnostico_anomalias.empty:
    if DIAG_CSV.exists():
        DIAG_CSV.unlink()
    print("No se detectan anomalías en la ventana analítica 2023-2026.")
    print("No se genera CSV de diagnóstico de anomalías.")
else:
    diagnostico_anomalias.to_csv(DIAG_CSV, index=False)
    print(f"Diagnóstico guardado en: {DIAG_CSV}")
    display(diagnostico_anomalias)

print("\nResumen de calidad:")
print(f"- Checks críticos superados: {checks_criticos_ok}")
print(f"- Checks de aviso superados: {checks_aviso_ok}")

display(quality)


No se detectan anomalías en la ventana analítica 2023-2026.
No se genera CSV de diagnóstico de anomalías.

Resumen de calidad:
- Checks críticos superados: True
- Checks de aviso superados: True


,check,valor,ok,detalle,nivel
0,fechas_nulas,0,True,Debe ser 0 para poder usar la tabla como dimen...,critico
1,duplicados_por_fecha,0,True,Debe haber una única fila por fecha.,critico
2,cobertura_anios_2023_2026,"{""presentes"": [2023, 2024, 2025, 2026], ""falta...",True,La salida limpia debe cubrir exactamente la ve...,critico
3,fechas_faltantes_2023_2026,0,True,Primeras fechas faltantes: [],critico
4,coherencia_fecha_vs_dia_semana_raw,0,True,"Compara el día de semana declarado en raw, ya ...",critico
5,valores_unicos_tipo_dia_raw,"[""domingo"", ""festivo"", ""laborable"", ""sabado""]",True,"Debe revisarse que aparecen laborable, sábado,...",critico
6,domingos_marcados_como_domingo,0,True,Todo domingo real debería aparecer marcado com...,aviso
7,festivos_detectados,56,True,Debe haber festivos identificados si el calend...,critico
8,festivos_sin_tipo_ni_festividad,0,True,"Si es >0, hay festivos detectados solo por tip...",aviso
9,laborable_y_festivo_a_la_vez,0,True,Una fecha no debería ser laborable y festiva s...,critico


## 7. Resúmenes interpretables

Se revisa la distribución de días por año y tipo de régimen diario base.

Este bloque sirve para entender si el calendario tiene una cobertura razonable y si puede usarse como apoyo para validar `ser_tiques`.

In [117]:
resumen_tipo_dia = (
    clean
    .groupby(["anio", "tipo_regimen_ser_dia_base"], dropna=False)
    .size()
    .reset_index(name="n_dias")
    .sort_values(["anio", "tipo_regimen_ser_dia_base"])
)

display(resumen_tipo_dia)

,anio,tipo_regimen_ser_dia_base,n_dias
0,2023,domingo,53
1,2023,festivo,14
2,2023,laborable,246
3,2023,sabado,52
4,2024,domingo,52
5,2024,festivo,14
6,2024,laborable,251
7,2024,sabado,49
8,2025,domingo,52
9,2025,festivo,14


In [118]:
resumen_flags = pd.DataFrame({
    "metrica": [
        "n_filas",
        "fecha_min",
        "fecha_max",
        "n_fechas_unicas",
        "n_laborables",
        "n_sabados",
        "n_domingos",
        "n_festivos",
        "n_dias_potencial_ser_observable",
    ],
    "valor": [
        len(clean),
        clean["fecha"].min(),
        clean["fecha"].max(),
        clean["fecha"].nunique(),
        int(clean["es_laborable"].sum()),
        int(clean["es_sabado"].sum()),
        int(clean["es_domingo"].sum()),
        int(clean["es_festivo"].sum()),
        int(clean["es_dia_potencial_ser_observable"].sum()),
    ]
})

display(resumen_flags)

,metrica,valor
0,n_filas,1461
1,fecha_min,2023-01-01 00:00:00
2,fecha_max,2026-12-31 00:00:00
3,n_fechas_unicas,1461
4,n_laborables,995
5,n_sabados,208
6,n_domingos,209
7,n_festivos,56
8,n_dias_potencial_ser_observable,1196


### Lectura de resultados

Tras aplicar el filtro temporal, la tabla queda limitada a 2023–2026, con 1.461 fechas únicas. Esta cifra es coherente con cuatro años completos, incluyendo 2024 como año bisiesto.

Las validaciones críticas se superan: no hay fechas nulas, no hay duplicados por fecha, no faltan días dentro de la ventana analítica y el día de semana calculado desde la fecha coincide con el declarado en el fichero raw.

También se superan los avisos. Por tanto, no se genera un CSV adicional de anomalías.

La distribución por tipo de día muestra una estructura razonable: cada año mantiene 14 festivos, domingos y sábados en magnitudes esperables, y días laborables compatibles con el calendario anual. La tabla queda preparada para cruzarse posteriormente con `ser_tiques` por `fecha` y validar el régimen SER observable por fecha y hora.

## 8. Guardado de salidas

Se guarda la tabla limpia en `data/interim/contexto/contexto_calendario_laboral/`.

También se guarda un CSV de checks de calidad en `reports/review/`.

In [119]:
final_cols = [
    "fecha",
    "anio",
    "mes",
    "dia",
    "dia_semana_num",
    "dia_semana_nombre",
    "tipo_dia_raw",
    "es_laborable",
    "es_sabado",
    "es_domingo",
    "es_festivo",
    "tipo_festivo",
    "festividad",
    "es_dia_potencial_ser_observable",
    "tipo_regimen_ser_dia_base",
    "archivo_origen",
    "dataset_id",
]

clean_final = clean[final_cols].copy()
clean_final = clean_final.sort_values("fecha").reset_index(drop=True)

try:
    clean_final.to_parquet(OUT_PARQUET, index=False)
except ImportError as e:
    raise ImportError(
        "No se pudo guardar Parquet. Instala pyarrow en el entorno activo: "
        "python -m pip install pyarrow"
    ) from e

quality.to_csv(QUALITY_CSV, index=False)

print(f"Guardado parquet: {OUT_PARQUET}")
print(f"Guardado quality checks: {QUALITY_CSV}")
print(f"Filas finales: {len(clean_final):,}")

display(clean_final.head())

Guardado parquet: /Users/hugo/TFM_parking_madrid/data/interim/contexto/contexto_calendario_laboral/contexto_calendario_laboral_clean.parquet
Guardado quality checks: /Users/hugo/TFM_parking_madrid/reports/tables/contexto_calendario_laboral_quality_checks.csv
Filas finales: 1,461


,fecha,anio,mes,dia,dia_semana_num,dia_semana_nombre,tipo_dia_raw,es_laborable,es_sabado,es_domingo,es_festivo,tipo_festivo,festividad,es_dia_potencial_ser_observable,tipo_regimen_ser_dia_base,archivo_origen,dataset_id
0,2023-01-01,2023,1,1,7,domingo,domingo,False,False,True,False,<NA>,<NA>,False,domingo,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral
1,2023-01-02,2023,1,2,1,lunes,laborable,True,False,False,False,<NA>,<NA>,True,laborable,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral
2,2023-01-03,2023,1,3,2,martes,laborable,True,False,False,False,<NA>,<NA>,True,laborable,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral
3,2023-01-04,2023,1,4,3,miércoles,laborable,True,False,False,False,<NA>,<NA>,True,laborable,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral
4,2023-01-05,2023,1,5,4,jueves,laborable,True,False,False,False,<NA>,<NA>,True,laborable,data/raw/contexto/contexto_calendario_laboral/...,contexto_calendario_laboral


## 9. Interpretación automática de resultados

Se genera un markdown breve con la interpretación de cobertura, duplicados y utilidad metodológica de la fuente.

Este informe sirve para documentar la limpieza y trasladar conclusiones a la memoria o a `docs/`.

In [120]:

checks_criticos_ok = bool(quality.loc[quality["nivel"].eq("critico"), "ok"].all())
checks_aviso_ok = bool(quality.loc[quality["nivel"].eq("aviso"), "ok"].all())

fecha_min = clean_final["fecha"].min()
fecha_max = clean_final["fecha"].max()

n_dup_fecha = int(clean_final.duplicated(subset=["fecha"], keep=False).sum())
n_fecha_nula = int(clean_final["fecha"].isna().sum())

years_present = sorted(clean_final["anio"].dropna().astype(int).unique().tolist())
missing_years = sorted(set(expected_years) - set(years_present))

resumen_tipo_dia_md = df_to_markdown_safe(resumen_tipo_dia)

n_domingo_diag = len(diagnostico_domingo) if "diagnostico_domingo" in globals() else "no_calculado"
n_festivo_diag = len(diagnostico_festivo) if "diagnostico_festivo" in globals() else "no_calculado"
hay_diag_csv = DIAG_CSV.exists() if "DIAG_CSV" in globals() else False

if hay_diag_csv:
    diagnostico_txt = "- Diagnóstico de anomalías: `reports/tables/contexto_calendario_laboral_diagnostico_anomalias.csv`"
else:
    diagnostico_txt = "- Diagnóstico de anomalías: no generado, porque no hay anomalías en la ventana 2023-2026"

md_report = f"""# Interpretación — contexto_calendario_laboral

## Objetivo de la fuente

`contexto_calendario_laboral` se limpia como tabla temporal diaria auxiliar. No mide aparcamiento ni ocupación, pero permite segmentar días comparables y validar posteriormente cuándo la señal de tiques SER debería ser observable.

## Cobertura raw y cobertura analítica

El fichero raw cargado cubre una ventana más amplia, pero la salida limpia del TFM se restringe a 2023-2026.

- Fecha mínima de la salida limpia: `{fecha_min.date() if pd.notna(fecha_min) else "NA"}`
- Fecha máxima de la salida limpia: `{fecha_max.date() if pd.notna(fecha_max) else "NA"}`
- Años presentes en la salida limpia: `{years_present}`
- Años esperados para el TFM: `{expected_years}`
- Años faltantes en 2023-2026: `{missing_years}`
- Filas finales: `{len(clean_final)}`

## Calidad básica

- Fechas nulas: `{n_fecha_nula}`
- Duplicados por fecha: `{n_dup_fecha}`
- Checks críticos superados: `{checks_criticos_ok}`
- Checks de aviso superados: `{checks_aviso_ok}`

## Avisos detectados en 2023-2026

- Domingos reales no marcados explícitamente como `domingo` en `tipo_dia_raw`: `{n_domingo_diag}`
- Festivos sin `tipo_festivo` ni `festividad`: `{n_festivo_diag}`

Si ambos valores son 0, no hay anomalías relevantes dentro de la ventana analítica usada en el TFM.

## Distribución por tipo de régimen diario base

{resumen_tipo_dia_md}

## Interpretación metodológica

La fuente es apta para construir una dimensión diaria porque supera los checks críticos: no hay fechas nulas, no hay duplicados por fecha, cubre el periodo 2023-2026 y la fecha real es coherente con el día de semana declarado.

La tabla permite validar la ausencia o caída de tiques en domingos y festivos sin interpretar automáticamente esa ausencia como baja dificultad de aparcamiento. Para el régimen SER horario completo no basta esta tabla diaria: la validación fina debe hacerse después en `ser_tiques`, cruzando fecha y hora del tique.

## Salidas generadas

- Tabla limpia: `data/interim/contexto/contexto_calendario_laboral/contexto_calendario_laboral_clean.parquet`
- Checks de calidad: `reports/tables/contexto_calendario_laboral_quality_checks.csv`
{diagnostico_txt}
"""

INTERPRETACION_MD.write_text(md_report, encoding="utf-8")

print(f"Guardado markdown: {INTERPRETACION_MD}")
print(md_report)


Guardado markdown: /Users/hugo/TFM_parking_madrid/docs/limpieza/contexto_calendario_laboral.md
# Interpretación — contexto_calendario_laboral

## Objetivo de la fuente

`contexto_calendario_laboral` se limpia como tabla temporal diaria auxiliar. No mide aparcamiento ni ocupación, pero permite segmentar días comparables y validar posteriormente cuándo la señal de tiques SER debería ser observable.

## Cobertura raw y cobertura analítica

El fichero raw cargado cubre una ventana más amplia, pero la salida limpia del TFM se restringe a 2023-2026.

- Fecha mínima de la salida limpia: `2023-01-01`
- Fecha máxima de la salida limpia: `2026-12-31`
- Años presentes en la salida limpia: `[2023, 2024, 2025, 2026]`
- Años esperados para el TFM: `[2023, 2024, 2025, 2026]`
- Años faltantes en 2023-2026: `[]`
- Filas finales: `1461`

## Calidad básica

- Fechas nulas: `0`
- Duplicados por fecha: `0`
- Checks críticos superados: `True`
- Checks de aviso superados: `True`

## Avisos detectados en 2023

## Conclusión

La fuente `contexto_calendario_laboral` queda transformada en una tabla diaria limpia, con una fila por fecha y variables derivadas para distinguir laborables, sábados, domingos y festivos.

La tabla no se usa todavía para construir joins con SER, pero queda preparada para validar el régimen observable de `ser_tiques` y para segmentar análisis posteriores de SER, EMT, tráfico y clima.

La columna `es_dia_potencial_ser_observable` debe interpretarse con cautela: solo indica días donde puede existir régimen SER observable a nivel diario. El régimen horario completo se validará posteriormente usando las horas de los tiques.